[![Fixel Algorithms](https://i.imgur.com/AqKHVZ0.png)](https://fixelalgorithms.gitlab.io)

# AI Program

## Deep Learning - Image to Image - Image Segmentation with U-Net

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 04/09/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2024_02/0099DeepLearningObjectDetection.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Image Processing and Computer Vision

# Machine Learning
from sklearn.model_selection import train_test_split

# Deep Learning
import torch
import torch.nn            as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

import torchinfo

from torchmetrics.functional.image import structural_similarity_index_measure
from torchmetrics.functional.regression import r2_score

import torchvista

import torchvision
from torchvision.io import decode_image
from torchvision.transforms import v2 as TorchVisionTrns

# Miscellaneous
import os
from platform import python_version
import random
import time
from zipfile import ZipFile

# Typing
from typing import Callable, Dict, Generator, List, Literal, Optional, Self, Set, Tuple, Union
from numpy.typing import NDArray
from torch import Tensor

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

# Improve performance by benchmarking
torch.backends.cudnn.benchmark = True

# Reproducibility (Per PyTorch Version on the same device)
# torch.manual_seed(seedNum)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark     = False #<! Makes things slower

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

PROJECT_NAME       = 'FixelCourses'
DATA_FOLDER_NAME   = 'DataSets'
MODELS_FOLDER_NAME = 'Models'
BASE_FOLDER_PATH   = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH   = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)
MODELS_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, MODELS_FOLDER_NAME)

TENSOR_BOARD_BASE   = 'TB'

In [ ]:
# Download Auxiliary Modules for Google Colab
if runInGoogleColab:
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataManipulation.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataVisualization.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DeepLearningPyTorch.py

In [ ]:
# Courses Packages

from DataManipulation import DownloadUrl
from DeepLearningPyTorch import GenDataLoaders, GetBatch, TrainModel

In [ ]:
# General Auxiliary Functions

def TensorImageNumpy( tZ: Tensor ) -> NDArray:
    """
    Converts a PyTorch Tensor to a Numpy Array.
    """
    mZ = tZ.squeeze()
    mX = mZ.detach().cpu().numpy()

    return mX

def TensorImgNumpy( tI: Tensor ) -> NDArray:
    """
    Converts a PyTorch Tensor Image to a Numpy Array Image.
    """
    
    mI = TensorImageNumpy(tI.permute(1, 2, 0))

    return mI

def SSIMScore( tYHat: Tensor, tY: Tensor ) -> Tensor:
    """
    Computes the Structural Similarity Index Measure (SSIM) between two images.
    Assumes that the input images are in the range [0, 1].
    """
    return structural_similarity_index_measure(tYHat, tY, data_range = 1.0)

def ImageR2Score( tYHat: Tensor, tY: Tensor ) -> Tensor:
    """
    Computes the R2 score between two images.
    Assumes that the input images are in the range [0, 1].
    """
    return r2_score(tYHat.flatten(), tY.flatten(), multioutput = 'uniform_average')

## Image to Image Models

_Image to Image_ models (Also called `Pix2Pix` / _Image to Image Translation_) are models which transform the input image into an output image.  
There many applications of such models:

 - Feature Extraction    
   Extract edges, corners, Mask, etc...
 - Color Adjustment  
   Apply RGB to Gray / Gray to RGB transformations.  
   Adjust White Balance / Tonal Curve.
 - Styling  
   Style transfer, style application.
 - Modality Transformation  
   RGB to IR / RGB to SAR and vice versa.  
   Image to Map.
 - Deconvolution  
   Super Resolution, Deblurring, Denoising.

The applications drove many developments in the Deep Learning field:

 - Architectures  
   U-Net, ViT.
 - Training Approach / Loss  
   Variational Auto Encoder (VAE), Generative Adversarial Network (GAN), Diffusion Models.

### Image to Map

This notebook tries to generate a map from an aerial image.

![](https://i.imgur.com/302e0vE.png)
<!-- ![](https://i.postimg.cc/rpRxPWyK/AAA.png) -->

<!-- ![](https://i.postimg.cc/dVMyZgQj/AAA.png)
![](https://i.imgur.com/5C494ie.png) -->


</br>

* <font color='brown'>(**#**)</font> The U-Net model was original created in the context of medical imaging: [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597).
* <font color='brown'>(**#**)</font> Additional ideas for similar task:
  - Maps / Satellite Aerial Images (Pix2Pix Maps)
    Use aligned pair of aerial images and Google Map like maps.  
    Use U-Net Model to move from one to another.
  - Edge Map to Image  
    Use the [UT Zappos50K Dataset](https://vision.cs.utexas.edu/projects/finegrained/utzap50k).  
    Run Canny edge Detection on the image. Use a U-Net model to reproduce the the image from the edges.
  - Grayscale to Colorization  
    Use the CIFAR-10 images. Convert to grayscale and use a U-Net model to recover color from the grayscale image.
  - Labels to Image  
    Based on the [CMP Facade Database](https://cmp.felk.cvut.cz/~tylecr1/facade) recover the image from the labels.

In [ ]:
# Parameters

# Data
dataSet    = 'SatAerialToMap'
dataSetUrl = r'https://huggingface.co/datasets/Royi/DataSets/resolve/main/SatAerialToMap.zip'

imgSize = 256

trainSampleRatio = 0.9 #<! Ratio of training images to total images
valSampleRatio   = 1 - trainSampleRatio #<! Ratio of validation images to total images

# Model
modelName      = 'ModelImgSegSatMap_2026_09_05.pt' #<! OneDrive -> Courses -> Models -> AIProgram
numFiltersBase = 16
weightSeg      = 3.0
weightCls      = 1.0
segThr         = 0.5

# Training
lossType   = 'MSE'
scoreType  = 'R2'
batchSize  = 8
numWorkers = 0 #<! Number of workers
numEpochs  = 35

# Optimizer
ηOpt        = 1e-4        #<! Optimizer learning rate (Has no effect if using scheduler)
tuβ         = (0.9, 0.99) #<! Betas for ADAM like Optimizer
weightDecay = 5e-5        #<! Optimizer weight decay
ηSch        = 7.5e-5      #<! Scheduler learning rate

# Visualization
numImg = 3

## Generate / Load Data

The data is the [Mobile Phone Screen Surface Defect Segmentation Dataset (MSD)](https://github.com/jianzhang96/MSD).  

* <font color='brown'>(**#**)</font> The data is downloaded by the Kaggle version of the dataset: [Kaggle - Mobile Phone Screen Surface Defect Segmentation Dataset](https://www.kaggle.com/datasets/girish17019/mobile-phone-defect-segmentation-dataset).

In [ ]:
# Extract Files
# Will create:
# - `FixelCourses/DataSets/SatAerialToMap/Train` - Contains all images.
# - `FixelCourses/DataSets/SatAerialToMap/Validation` - Contains all images.
# Each image is (600, 1200, 3) where the left (600, 600, 3) is the aerial image and the right (600, 600, 3) is the map image.

datasetFolderPath     = os.path.join(DATA_FOLDER_PATH, dataSet)

# Delete existing folders if any
if not os.path.isdir(datasetFolderPath):
    # 1. Download the ZIP file by URL.
    # 2. Extract the ZIP file to the dataset folder path.
    # 3. Delete the ZIP file.

    fileName = os.path.join(DATA_FOLDER_PATH, f'{dataSet}.zip')
    DownloadUrl(dataSetUrl, DATA_FOLDER_PATH)
    with ZipFile(fileName, 'r') as zipFile:
        zipFile.extractall(DATA_FOLDER_PATH) #<! The Zip file contains a folder
    time.sleep(1.0) #<! Wait for the file system to update
    os.remove(fileName)

* <font color='red'>(**?**)</font> Go through files using the OS's image viewer. Specifically the mask images. What can you say about the classes per image?

<!-- Each image mask contain only a single class. Hence predicting the mask class can be done in global manner and not in a per pixel manner. -->

### DataSet

Generate a `DataSet` class as a loader of the data.

* <font color='brown'>(**#**)</font> Since each image contains a single class, the mask can be represented as a binary mask for the defect area and a global class label by a classification head.
* <font color='brown'>(**#**)</font> There images with no defects. Hence a `None` class should be added.

In [ ]:
# The Dataset Class

class SatAerialMapDataset(Dataset):
    def __init__( self, rootFolderPath: str, dataSet: Literal['Train', 'Validation', 'All'], /, *, imgSize: Optional[int] = None, hTrns: Optional[Callable] = None ) -> None:
        """
        Satellite Aerial Map Segmentation Dataset.
        The dataset folder structure:
         - Train
            - 00001.jpg
            - 00002.jpg
            - ...
         - Validation
            - 00001.jpg
            - 00002.jpg
            - ...
        Each image is (600, 1200, 3) where the left (600, 600, 3) is the aerial image and the right (600, 600, 3) is the map image.

        Parameters
        ----------
        rootFolderPath : str
            Path to the folder containing images sets.
        dataSet : Literal['Train', 'Validation', 'All']
            Dataset type to be used. Can be 'Train', 'Validation', or 'All'.
        hTrns : Optional[Callable], optional
            Transform to be applied on the features image (Aerial).
            Should be limited to pixel wise transforms (e.g., normalization, color jitter, etc...).
            By default None.
        """
        super().__init__()

        if dataSet not in ('Train', 'Validation', 'All'):
            raise ValueError("dataSet must be 'Train', 'Validation', or 'All'")

        lDataSets = ['Train', 'Validation'] if dataSet == 'All' else [dataSet]
        lImgFiles = []
        for dataSetName in lDataSets:
            dataSetFolderPath = os.path.join(rootFolderPath, dataSetName)
            if not os.path.isdir(dataSetFolderPath):
                raise FileNotFoundError(f'Dataset folder does not exist: {dataSetFolderPath}')

            lDataSetFiles = sorted(
                os.path.join(dataSetFolderPath, fileName)
                for fileName in os.listdir(dataSetFolderPath)
                if os.path.isfile(os.path.join(dataSetFolderPath, fileName)) and fileName.lower().endswith(('.jpg', '.jpeg', '.png'))
            )
            lImgFiles.extend(lDataSetFiles)

        self._lImgFiles = lImgFiles
        self._imgSize   = imgSize
        self._hTrns     = hTrns

    def __len__( self ) -> int:
        """
        Returns the number of paired images in the dataset.
        """

        return len(self._lImgFiles)

    def __getitem__( self, idx: int ) -> Tuple[Tensor, Tensor]:
        """
        Returns the aerial image and its corresponding map image.

        Parameters
        ----------
        idx : int
            Index of the sample to be fetched.

        Returns
        -------
        Tuple[Tensor, Tensor]
            A tuple containing the aerial image and the map image.
        """
        tPair = decode_image(self._lImgFiles[idx], mode = 'RGB')
        imgWidth = tPair.shape[2]

        imgWidthHalf = imgWidth // 2
        tX = tPair[:, :, :imgWidthHalf]
        tY = tPair[:, :, imgWidthHalf:]

        if self._imgSize is not None:
            tX = TorchVisionTrns.functional.resize(tX, size = (self._imgSize, self._imgSize))
            tY = TorchVisionTrns.functional.resize(tY, size = (self._imgSize, self._imgSize), interpolation = TorchVisionTrns.InterpolationMode.NEAREST)

        if self._hTrns:
            tX = self._hTrns(tX)

        tY = TorchVisionTrns.functional.to_dtype(tY, torch.float, scale = True)

        return tX, tY

    def SetImageSize( self, imgSize: Optional[int] ) -> None:
        """
        Sets the image size for resizing the aerial and map images.

        Parameters
        ----------
        imgSize : Optional[int]
            The desired image size. If None, no resizing will be applied.
        """
        self._imgSize = imgSize

    def SetTransforms( self, hTrns: Optional[Callable] ) -> None:
        """
        Sets the pixel wise transforms applied to the aerial image.
        """
        self._hTrns = hTrns

In [ ]:
# The Dataset

dsData     = SatAerialMapDataset(datasetFolderPath, 'All', imgSize = imgSize, hTrns = None)
numSamples = len(dsData)

print(f'Number of samples in the dataset: {numSamples}')

### Plot the Data

In [ ]:
# Plot Random Samples from the Dataset

rndIdx = random.randint(0, numSamples - 1)
tX, tY = dsData[rndIdx]

tX = TensorImgNumpy(tX)
tY = TensorImgNumpy(tY)

hF, vHa = plt.subplots(1, 2, figsize = (8, 4))
vHa = vHa.flat

hA = vHa[0]
hA.imshow(tX)
hA.set_title('Image')
hA.axis('off');

hA = vHa[1]
hA.imshow(tY)
hA.set_title(f'Target')
hA.axis('off');

* <font color='red'>(**?**)</font> How would you model the target image? Would you use a Classification or Regression model?

In [ ]:
# Dataset Transform

oTrnsTrain = TorchVisionTrns.Compose([
    # TorchVisionTrns.ToImage(),
    TorchVisionTrns.ToDtype(torch.float, scale = True),
    TorchVisionTrns.RandomChoice([
            TorchVisionTrns.RandomGrayscale(p = 1.0),
            TorchVisionTrns.GaussianBlur(7, sigma = (0.1, 1.0)),
            TorchVisionTrns.RandomEqualize(p = 1.0),
            TorchVisionTrns.RandomAutocontrast(p = 1.0),
            TorchVisionTrns.GaussianNoise(sigma = 0.05),
            TorchVisionTrns.RandomErasing(p = 1.0, scale = (0.05, 0.15), ratio = (0.5, 2.0), value = 0, inplace = True),
            TorchVisionTrns.RGB(), #<! Identity for RGB Images
        ], p = [0.07, 0.07, 0.07, 0.07, 0.07, 0.07, 0.58]),
])

oTrnsVal = TorchVisionTrns.Compose([
    # TorchVisionTrns.ToImage(),
    TorchVisionTrns.ToDtype(torch.float, scale = True),
])

dsData.SetTransforms(oTrnsTrain)

* <font color='blue'>(**!**)</font> Add color related augmenation.

In [ ]:
# Plot Random Samples from the Dataset

rndIdx = random.randint(0, numSamples - 1)
tX, tY = dsData[rndIdx]

tX = TensorImgNumpy(tX)
tY = TensorImgNumpy(tY)

hF, vHa = plt.subplots(1, 2, figsize = (8, 4))
vHa = vHa.flat

hA = vHa[0]
hA.imshow(tX)
hA.set_title('Image')
hA.axis('off');

hA = vHa[1]
hA.imshow(tY)
hA.set_title(f'Target')
hA.axis('off');

In [ ]:
# Create Training and Validation Datasets

# Creating 2 separate datasets for training and validation allows us to apply different transformations to each dataset
dsTrain = SatAerialMapDataset(datasetFolderPath, 'All', imgSize = imgSize, hTrns = oTrnsTrain)
dsVal   = SatAerialMapDataset(datasetFolderPath, 'All', imgSize = imgSize, hTrns = oTrnsVal)

vIdxTrain, vIdxVal = train_test_split(np.arange(numSamples), test_size = valSampleRatio, train_size = trainSampleRatio, random_state = seedNum, shuffle = True)

dsTrain = torch.utils.data.Subset(dsTrain, vIdxTrain)
dsVal   = torch.utils.data.Subset(dsVal, vIdxVal)

print(f'The training data set contains  : {len(dsTrain):4d} samples.')
print(f'The validation data set contains: {len(dsVal):4d} samples.')

* <font color='brown'>(**#**)</font> One could use negative values for the bounding box. The model will extrapolate the object dimensions.

In [ ]:
# Data Loader

dlTrain, dlVal = GenDataLoaders(dsTrain, dsVal, batchSize, numWorkers = numWorkers, persWork = False)

* <font color='red'>(**?**)</font> Why are lists used instead of arrays for the labels and the bounding boxes?

In [ ]:
# Element of the Data Set / Data Sample

tX, tY = dsTrain[0]

print(f'The features shape: {tX.shape}')
print(f'The features type : {tX.dtype}')
print(f'The labels shape  : {tY.shape}')
print(f'The labels type   : {tY.dtype}')

In [ ]:
# Element of the Dataloader

tX, tY = GetBatch(dlTrain)

print(f'The batch of features shape: {tX.shape}')
print(f'The batch of features type : {tX.dtype}')
print(f'The batch of labels shape  : {tY.shape}')
print(f'The batch of labels type   : {tY.dtype}')

* <font color='brown'>(**#**)</font> Since the labels are in the same contiguous container as the bounding box parameters, their type is `Float`.
* <font color='brown'>(**#**)</font> The bounding box is using absolute values. In practice it is commonly normalized to the image dimensions.

## The Model

The U-Net Models commonly use the Transposed Convolution layer or the Upsampling Layer.

### Upsample (Zero Insertion) and Convolution (Learned LPF)

A transposed convolution can be interpreted as two consecutive operations:

1. **Upsample by Zero Insertion:** insert zeros between adjacent spatial samples according to the stride.
2. **Apply a Learned Convolution:** replace the fixed low pass filter (LPF) used in classical interpolation with a kernel learned from data.

```mermaid
flowchart LR
    A["Input feature map<br/>Low spatial resolution"] --> B["Zero-insertion upsampling<br/>Insert zeros between samples"]
    B --> C["Sparse upsampled feature map<br/>Higher spatial resolution"]
    C --> D["Learned convolution<br/>Trainable LPF-like kernel"]
    D --> E["Dense output feature map<br/>Learned interpolation"]
```

The convolution spreads each original sample over its neighborhood and fills the inserted locations.  
Calling it an LPF describes the interpolation viewpoint, because its coefficients are learned, it is not guaranteed to be strictly low pass.

* <font color='brown'>(**#**)</font> PyTorch implements the combined operation efficiently with [`ConvTranspose2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html), without explicitly allocating a tensor containing the inserted zeros.
* <font color='brown'>(**#**)</font> With compatible stride, padding, kernel orientation, and boundary handling, transposed convolution is equivalent to zero insertion followed by convolution.
* <font color='brown'>(**#**)</font> `output_padding` in `ConvTranspose2d` resolves output shape ambiguity.
* <font color='brown'>(**#**)</font> Uneven kernel overlap, especially when the kernel size is not divisible by the stride, can produce checkerboard artifacts. Explicit resize using nearest-neighbor or bilinear interpolation followed by a standard convolution often reduces this effect, but it is a different operation from zero insertion followed by convolution.
* <font color='brown'>(**#**)</font> Transposed convolution learns the upsampling rule jointly and may be more expressive. Resize followed by convolution separates geometric resizing from feature filtering and often gives more predictable spatial behavior.
* <font color='brown'>(**#**)</font> Further reading: [Distill - Deconvolution and Checkerboard Artifacts](https://distill.pub/2016/deconv-checkerboard/), [Convolution Arithmetic](https://github.com/vdumoulin/conv_arithmetic) and [A guide to convolution arithmetic for deep learning](https://arxiv.org/abs/1603.07285).

### Interpolation (Resize) and Convolution (Learned LPF)

Resize followed by convolution can be interpreted as two consecutive operations:

1. **Upsample by Interpolation**: Increase the spatial resolution using a fixed interpolation rule, such as _Nearest Neighbor_ or _BiLinear_ interpolation.
2. **Apply a Learned Convolution**: Filter and refine the dense upsampled feature map using a kernel learned from data.

```mermaid
flowchart LR
    A["Input feature map<br/>Low spatial resolution"] --> B["Fixed interpolation<br/>Nearest-neighbor or bilinear"]
    B --> C["Dense upsampled feature map<br/>Higher spatial resolution"]
    C --> D["Learned convolution<br/>Trainable LPF-like kernel"]
    D --> E["Refined output feature map<br/>Learned feature reconstruction"]
```

Unlike zero insertion (_Expansion_), interpolation directly assigns a value to every location in the upsampled feature map.
The following convolution learns how to combine neighboring values, reduce interpolation artifacts and refine spatial details. Calling it an LPF describes its interpolation role; because its coefficients are learned, it is not guaranteed to be strictly low pass.

* <font color='brown'>(#)</font> _Nearest Neighbor_ interpolation copies existing values and preserves sharp transitions, but may produce blocky features. 
* <font color='brown'>(#)</font> _BiLinear_ interpolation computes weighted averages of neighboring values and generally produces smoother features.
* <font color='brown'>(#)</font> The interpolation rule is fixed and has no learned parameters. The subsequent convolution adapts to the training data.
* <font color='brown'>(#)</font> Since every output location is populated before convolution, spatial coverage is more uniform than with zero insertion.  
This often reduces checkerboard artifacts caused by uneven kernel overlap.
* <font color='brown'>(#)</font> PyTorch implements the two operations using `Upsample` or `interpolate()`, followed by `Conv2d`.
* <font color='brown'>(#)</font> When using _BiLinear_ interpolation for feature maps, `align_corners = False` is commonly used because the sampling locations remain consistent across different input sizes.

### Comparison of Upsampling Approaches

| Approach | Operation | Learned Component | Advantages | Disadvantages |
|----------|-----------|-------------------|------------|---------------|
| **Transposed Convolution** | Zero insertion followed by convolution | Complete upsampling filter | Flexible and fully learned; can combine resizing and feature extraction efficiently; | May produce checkerboard artifacts due to uneven kernel overlap; output size can be less intuitive; |
| **Explicit Resize** | Nearest Neighbor or BiLinear interpolation | None | Simple, fast, parameter free and predictable; output size is explicit; | Fixed interpolation cannot adapt to the data; Nearest Neighbor may look blocky and BiLinear may smooth boundaries; |
| **Explicit Resize + Convolution** | Fixed interpolation followed by standard convolution | Feature refinement after resizing | Predictable spatial coverage with learned refinement; usually reduces checkerboard artifacts; | The resize rule remains fixed; the dense intermediate feature map may increase memory traffic; |

* <font color='brown'>(**#**)</font> PyTorch provides these approaches through [`ConvTranspose2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ConvTranspose2d.html), [`Upsample`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html) or [`interpolate`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.interpolate.html) and an upsampling operation followed by `Conv2d`, respectively.

In [ ]:
# Inverted Residual Block

class InvertedResidualBlock(nn.Module):
    """
    Modern block structure: Expand -> Depth Wise Convolution -> Project.
    Includes a skip connection if input and output shapes match.
    """
    def __init__(self, numChnlIn: int, numChnlOut: int, expFctr: int = 4, strideSize: int = 1):
        super().__init__()
        
        self.strideSize = strideSize
        self.enableSkip = (strideSize == 1 and numChnlIn == numChnlOut)
        hiddenDim       = numChnlIn * expFctr

        self.oBlock = nn.Sequential(
            # Expansion of Channels (Using 1x1 Convolution)
            nn.Conv2d(numChnlIn, hiddenDim, 1, bias = False),
            nn.BatchNorm2d(hiddenDim),
            nn.SiLU(),
            
            # Depth Wise Convolution (3x3)
            nn.Conv2d(hiddenDim, hiddenDim, 3, stride = strideSize, padding = 1, groups = hiddenDim, bias = False),
            nn.BatchNorm2d(hiddenDim),
            nn.SiLU(),
            
            # Projection (Using 1x1 Convolution) - Linear bottleneck (No activation at end)
            nn.Conv2d(hiddenDim, numChnlOut, 1, bias = False),
            nn.BatchNorm2d(numChnlOut),
        )

    def forward(self, tX: Tensor) -> Tensor:
        
        if self.enableSkip:
            return tX + self.oBlock(tX)
        else:
            return self.oBlock(tX)

In [ ]:
# Depthwise Separable Convolution Block

class DepthwiseSeparableConv(nn.Module):
    """
    The building block of efficient networks.
    Splits a standard convolution into:
    1. Depthwise: Spatial filtering (lightweight)
    2. Pointwise: Channel mixing (1x1 conv)
    """
    def __init__(self, numChnlIn: int, numChnlOut: int, strideSize: int = 1):
        super().__init__()
        
        self.strideSize = strideSize
        
        # Depthwise Convolution (3x3) (Each kernel per channel)
        self.oBlock001 = nn.Sequential(
            nn.Conv2d(numChnlIn, numChnlIn, kernel_size = 3, padding = 1,  stride = strideSize, groups = numChnlIn, bias = False),
            nn.BatchNorm2d(numChnlIn),
            nn.SiLU(), #<! Modern activation (Swish)
        )
        # Pointwise Convolution (1x1) Projection over channels
        self.oBlock002 = nn.Sequential(
            nn.Conv2d(numChnlIn, numChnlOut, kernel_size = 1, bias = False),
            nn.BatchNorm2d(numChnlOut),
            nn.SiLU(), #<! Modern activation (Swish)
        )

    def forward(self, tX: Tensor) -> Tensor:
        
        tX = self.oBlock001(tX)
        tX = self.oBlock002(tX)
        
        return tX

### Image to Image (Pix2Pix) Model

The task is to generate a map image from its aligned aerial image.  
The input and target are spatially aligned RGB images.
The problem is modeled as dense image regression: Per input pixel neighborhood, the model predicts the corresponding map color.

The U-Net structure:

1. **Encoder**: Extracts features at progressively lower spatial resolutions.
2. **Bottleneck**: Represents high level spatial and semantic information.
3. **Decoder**: Restores the original image resolution.
4. **Skip Connections**: Preserves fine spatial details from the aerial image.
5. **RGB Image Head**: Generate the image with values in the range $[0, 1]$.

The model outputs an RGB image with the same spatial dimensions as the input.

In [ ]:
# U-Net: Encoder and Decoder Blocks

class µUNet(nn.Module):
    def __init__( self, numChnlIn: int, numChnlOut: int = 3, numFiltersBase: int = 32 ) -> None:
        super().__init__()

        # Feature Extractor
        self.oFeatExt = nn.Sequential(
            nn.Conv2d(numChnlIn, numFiltersBase, 3, padding = 1, stride = 1, bias = False),
            nn.BatchNorm2d(numFiltersBase),
            nn.SiLU(),
        )

        # Encoder Stages
        self.oEnc001 = InvertedResidualBlock(numFiltersBase    , numFiltersBase * 2, strideSize = 2) #<! H/2
        self.oEnc002 = InvertedResidualBlock(numFiltersBase * 2, numFiltersBase * 4, strideSize = 2) #<! H/4
        self.oEnc003 = InvertedResidualBlock(numFiltersBase * 4, numFiltersBase * 8, strideSize = 2) #<! H/8
        self.oEnc004 = InvertedResidualBlock(numFiltersBase * 8, numFiltersBase * 16, strideSize = 2) #<! H/16

        # Embedding / Bottleneck
        self.oEmbed = InvertedResidualBlock(numFiltersBase * 16, numFiltersBase * 16, strideSize = 1) #<! H/16

        # Decoder (Explicit Resize + Convolution)
        self.oUp004 = nn.Sequential(
            nn.Upsample(scale_factor = 2, mode = 'bilinear', align_corners = False),
            nn.Conv2d(numFiltersBase * 16, numFiltersBase * 16, 3, padding = 1, bias = False),
            nn.BatchNorm2d(numFiltersBase * 16),
            nn.SiLU(),
        )
        self.oDec004 = nn.Sequential(
            nn.Conv2d(numFiltersBase * 16 + numFiltersBase * 8, numFiltersBase * 8, 1),
            InvertedResidualBlock(numFiltersBase * 8, numFiltersBase * 8),
        )

        self.oUp003 = nn.Sequential(
            nn.Upsample(scale_factor = 2, mode = 'bilinear', align_corners = False),
            nn.Conv2d(numFiltersBase * 8, numFiltersBase * 8, 3, padding = 1, bias = False),
            nn.BatchNorm2d(numFiltersBase * 8),
            nn.SiLU(),
        )
        self.oDec003 = nn.Sequential(
            nn.Conv2d(numFiltersBase * 8 + numFiltersBase * 4, numFiltersBase * 4, 1),
            InvertedResidualBlock(numFiltersBase * 4, numFiltersBase * 4),
        )

        self.oUp002 = nn.Sequential(
            nn.Upsample(scale_factor = 2, mode = 'bilinear', align_corners = False),
            nn.Conv2d(numFiltersBase * 4, numFiltersBase * 4, 3, padding = 1, bias = False),
            nn.BatchNorm2d(numFiltersBase * 4),
            nn.SiLU(),
        )
        self.oDec002 = nn.Sequential(
            nn.Conv2d(numFiltersBase * 4 + numFiltersBase * 2, numFiltersBase * 2, 1),
            InvertedResidualBlock(numFiltersBase * 2, numFiltersBase * 2),
        )

        self.oUp001 = nn.Sequential(
            nn.Upsample(scale_factor = 2, mode = 'bilinear', align_corners = False),
            nn.Conv2d(numFiltersBase * 2, numFiltersBase * 2, 3, padding = 1, bias = False),
            nn.BatchNorm2d(numFiltersBase * 2),
            nn.SiLU(),
        )

        # RGB Image Generation Head (C, H, W) in the [0, 1] range
        self.oHeadImg = nn.Sequential(
            nn.Conv2d(numFiltersBase * 2 + numFiltersBase, numFiltersBase * 2, 3, padding = 1, bias = False),
            nn.BatchNorm2d(numFiltersBase * 2),
            nn.SiLU(),
            nn.Conv2d(numFiltersBase * 2, numChnlOut, 1),
            nn.Sigmoid(),
        )

    def forward( self, tX: Tensor ) -> Tensor:
        # Assumes the input tensor tX is of shape (B, C, H, W) where `H` and `W` are divisible by 16.
        # Encoder
        tX0 = self.oFeatExt(tX) #<! H
        tX1 = self.oEnc001(tX0) #<! H/2
        tX2 = self.oEnc002(tX1) #<! H/4
        tX3 = self.oEnc003(tX2) #<! H/8
        tX4 = self.oEnc004(tX3) #<! H/16

        # Low Dimensional Embedding / Bottleneck
        tEm = self.oEmbed(tX4) #<! H/16

        # Decoder
        tD4 = self.oUp004(tEm)
        tD4 = self.oDec004(torch.cat([tD4, tX3], dim = 1))

        tD3 = self.oUp003(tD4)
        tD3 = self.oDec003(torch.cat([tD3, tX2], dim = 1))

        tD2 = self.oUp002(tD3)
        tD2 = self.oDec002(torch.cat([tD2, tX1], dim = 1))

        tD1 = self.oUp001(tD2)

        tY = self.oHeadImg(torch.cat([tD1, tX0], dim = 1))

        return tY

* <font color='red'>(**?**)</font> What is the motivation for the use of `Sigmoid` layer in the image head?

In [ ]:
# Model

oModel = µUNet(numChnlIn = tX.shape[1], numChnlOut = tY.shape[1], numFiltersBase = numFiltersBase)

In [ ]:
# Run device

runDevice = torch.device('cuda:0' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')) #<! The 1st CUDA device
print(f'Running on device: {runDevice}')

In [ ]:
# Model Summary

torchinfo.summary(oModel, tX.shape, col_names = ['input_size', 'output_size', 'num_params', 'kernel_size'], device = runDevice, row_settings = ['depth', 'var_names'])

In [ ]:
# Model Architecture

torchvista.trace_model(oModel.eval(), tX.to(runDevice))

## Train the Model



In [ ]:
# Loss Class

class Pix2PixLoss(nn.Module):
    def __init__( self, lossType: Literal['L1', 'SmoothL1', 'L2', 'MSE'] = 'MSE' ) -> None:
        """
        Pixel wise loss for image to image regression.

        Parameters
        ----------
        lossType : Literal['L1', 'SmoothL1', 'L2', 'MSE'], optional
            Loss function to use, by default 'MSE'.
        """
        super().__init__()

        match lossType:
            case 'L1':
                self.oLoss = nn.L1Loss()
            case 'SmoothL1':
                self.oLoss = nn.SmoothL1Loss()
            case 'L2' | 'MSE':
                self.oLoss = nn.MSELoss()
            case _:
                raise ValueError('The parameter `lossType` must be either `L1`, `SmoothL1`, `L2` or `MSE`')

    def forward( self, tYHat: Tensor, tY: Tensor ) -> Tensor:
        """
        Computes the pixel wise loss between the generated and target images.

        Parameters
        ----------
        tYHat : Tensor
            Generated image tensor (B x C x H x W).
        tY : Tensor
            Target image tensor (B x C x H x W).

        Returns
        -------
        Tensor
            Scalar image reconstruction loss.
        """

        return self.oLoss(tYHat, tY)

In [ ]:
# Score Class

class Pix2PixScore(nn.Module):
    def __init__( self, scoreType: Literal['SSIM', 'R2'] = 'SSIM' ) -> None:
        """
        Image quality score for image to image regression.

        Parameters
        ----------
        scoreType : Literal['SSIM', 'R2'], optional
            Score function to use, by default 'SSIM'.
        """
        super().__init__()

        match scoreType:
            case 'SSIM':
                self.hScore = SSIMScore
            case 'R2':
                self.hScore = ImageR2Score
            case _:
                raise ValueError('The parameter `scoreType` must be either `SSIM` or `R2`')

    def forward( self, tYHat: Tensor, tY: Tensor ) -> Tensor:
        """
        Computes the selected score between the generated and target images.

        Parameters
        ----------
        tYHat : Tensor
            Generated image tensor (B x C x H x W).
        tY : Tensor
            Target image tensor (B x C x H x W).

        Returns
        -------
        Tensor
            Scalar image quality score.
        """

        return self.hScore(tYHat, tY)

In [ ]:
# Loss and Score

hL = Pix2PixLoss(lossType = lossType)
hS = Pix2PixScore(scoreType = scoreType)
hL = hL.to(runDevice)
hS = hS.to(runDevice)

In [ ]:
# Optimizer Related

oOpt = torch.optim.AdamW(oModel.parameters(), lr = ηOpt, betas = tuβ, weight_decay = weightDecay) #<! Define optimizer
oSch = torch.optim.lr_scheduler.OneCycleLR(oOpt, max_lr = ηSch, total_steps = numEpochs)

In [ ]:
# Training Loop

oModel = oModel.to(runDevice)
_, lTrainLoss, lTrainScore, lValLoss, lValScore, lLearnRate = TrainModel(oModel, dlTrain, dlVal, oOpt, numEpochs, hL, hS, oSch = oSch)

In [ ]:
# Plot Training Phase

hF, vHa = plt.subplots(nrows = 1, ncols = 3, figsize = (18, 5))
vHa = np.ravel(vHa)

hA = vHa[0]
hA.plot(lTrainLoss, lw = 2, label = 'Train')
hA.plot(lValLoss, lw = 2, label = 'Validation')
hA.set_title(f'Loss')
hA.set_xlabel('Epoch')
hA.set_ylabel('Loss')
hA.legend()

hA = vHa[1]
hA.plot(lTrainScore, lw = 2, label = 'Train')
hA.plot(lValScore, lw = 2, label = 'Validation')
hA.set_title('Score')
hA.set_xlabel('Epoch')
hA.set_ylabel('Score')
hA.legend()

hA = vHa[2]
hA.plot(lLearnRate, lw = 2)
hA.set_title('Learn Rate Scheduler')
hA.set_xlabel('Epoch')
hA.set_ylabel('Learn Rate');

In [ ]:
# Load the Model
modelPath = os.path.join(MODELS_FOLDER_PATH, modelName)
if os.path.isfile(modelPath):
    dModel = torch.load(modelPath, map_location = runDevice)
    oModel.load_state_dict(dModel['Model'])
    print(f'Model loaded from: {modelPath}')

In [ ]:
# Process Sample through the Trained Model

rndIdx = random.randint(0, len(dsVal) - 1)
# tX, tY = dsTrain[rndIdx]
tX, tY = dsVal[rndIdx]

oModel.eval()
with torch.inference_mode():
    tXb = tX.unsqueeze(0).to(runDevice) #<! Add batch dimension
    tYHat = oModel(tXb).squeeze(0).cpu() #<! Remove batch dimension

tX    = TensorImgNumpy(tX)
tY    = TensorImgNumpy(tY)
tYHat = TensorImgNumpy(tYHat)

hF, vHa = plt.subplots(1, 3, figsize = (12, 4))
vHa = vHa.flat

hA = vHa[0]
hA.imshow(tX)
hA.set_title('Aerial Image')
hA.axis('off');

hA = vHa[1]
hA.imshow(tY)
hA.set_title('Target Map')
hA.axis('off');

hA = vHa[2]
hA.imshow(tYHat)
hA.set_title('Generated Map')
hA.axis('off');

* <font color='red'>(**?**)</font> Which post process operation can be done to improve results?
* <font color='green'>(**@**)</font> Adjust the hyper parameters to improve the results. In particular, try the `L1` loss, which often produces sharper images than the `MSE` loss.
* <font color='green'>(**@**)</font> Add a perceptual loss to encourage the generated map to match the target at a semantic and structural level. Instead of comparing only pixel values, compare intermediate feature maps extracted by a pretrained image classification network such as VGG-19. This loss can be combined with the pixel wise reconstruction loss when training the U-Net generator.  
See [Perceptual Losses for Real Time Style Transfer and Super Resolution](https://arxiv.org/abs/1603.08155), [The Unreasonable Effectiveness of Deep Features as a Perceptual Metric](https://arxiv.org/abs/1801.03924), [PerceptualSimilarity - LPI Metric](https://github.com/richzhang/PerceptualSimilarity), [ConvNeXt Perceptual Loss](https://github.com/sypsyp97/convnext_perceptual_loss).
* <font color='brown'>(**#**)</font> The main cause of blurred results is the use of an MSE based loss. When several plausible outputs exist, minimizing MSE encourages the model to predict their conditional mean, which averages fine details. Generative models use different / additional objectives (Often favoring the mode of the conditional output distribution) that better capture perceptual and distributional properties to produce sharper results.